In [3]:
# --- Setup: install the current Gemini SDK (google-genai, not the deprecated google-generativeai) ---
!pip install -q -U google-genai

from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

# Quick connection test with the current model name
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents="Say 'connection successful' if you can read this."
)
print(response.text)

connection successful


In [5]:
# --- Reload everything needed to generate a fresh snapshot in this notebook ---
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import joblib
import sys

# Load the cleaned dataset
df = pd.read_csv('/content/drive/MyDrive/fred-economic-agent/data/processed/economic_dataset.csv',
                  index_col='Date', parse_dates=True)

# Load the trained model + its expected feature list
final_model_v2 = joblib.load('/content/drive/MyDrive/fred-economic-agent/models/recession_model.pkl')
final_features_v2 = joblib.load('/content/drive/MyDrive/fred-economic-agent/models/recession_model_features.pkl')

# Load the reusable snapshot function from src/
sys.path.append('/content/drive/MyDrive/fred-economic-agent/src')
from snapshot import generate_economic_snapshot

# Generate the snapshot
snapshot = generate_economic_snapshot(df, final_model_v2, final_features_v2)

import json
print(json.dumps(snapshot, indent=2))

Mounted at /content/drive
{
  "date": "2026-07-01",
  "recession_risk": "LOW",
  "recession_probability": 0.183,
  "inflation_trend": "RISING",
  "inflation_rate": 3.3,
  "unemployment_trend": "FALLING",
  "unemployment_rate": 4.1,
  "industrial_production_trend": "RISING",
  "yield_curve_status": "NORMAL",
  "yield_curve_spread": 0.38
}


In [6]:
# --- Generate the plain-English explanation from the snapshot ---
def explain_economic_snapshot(snapshot, client):
    prompt = f"""You are an economic analyst assistant. You will be given a structured
economic snapshot with real, verified data. Your job is ONLY to explain and interpret
these exact numbers in plain business language — you must NOT invent any statistics,
numbers, or claims that are not present in the data below.

ECONOMIC SNAPSHOT (as of {snapshot['date']}):
- Recession Risk: {snapshot['recession_risk']} (model probability: {snapshot['recession_probability']*100:.1f}%)
- Inflation: {snapshot['inflation_rate']}%, trend is {snapshot['inflation_trend']}
- Unemployment: {snapshot['unemployment_rate']}%, trend is {snapshot['unemployment_trend']}
- Industrial Production trend: {snapshot['industrial_production_trend']}
- Yield Curve: {snapshot['yield_curve_status']} (spread: {snapshot['yield_curve_spread']} percentage points)

Write a short, clear explanation (4-6 sentences) covering:
1. Overall recession risk assessment
2. The 2-3 key indicators driving that assessment
3. What a business leader should take away from this

Do not use markdown headers. Write it as flowing paragraph(s), business-report tone.
"""
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt
    )
    return response.text

explanation = explain_economic_snapshot(snapshot, client)
print(explanation)

The overall risk of economic recession is currently low, supported by a model-estimated probability of just 18.3%. This favorable assessment is driven by strong output and labor market indicators, notably a falling unemployment rate of 4.1% and a rising trend in industrial production, alongside a normal yield curve spread of 0.38 percentage points. However, inflation currently stands at 3.3% and continues on a rising trend. For business leaders, the takeaway is an environment characterized by ongoing demand and solid fundamental activity, though planning should incorporate strategies to manage the cost implications of accelerating inflation.


In [7]:
# --- Save explanation function to src/genai_explain.py ---
explain_code = '''
def explain_economic_snapshot(snapshot, client):
    prompt = f"""You are an economic analyst assistant. You will be given a structured
economic snapshot with real, verified data. Your job is ONLY to explain and interpret
these exact numbers in plain business language — you must NOT invent any statistics,
numbers, or claims that are not present in the data below.

ECONOMIC SNAPSHOT (as of {snapshot['date']}):
- Recession Risk: {snapshot['recession_risk']} (model probability: {snapshot['recession_probability']*100:.1f}%)
- Inflation: {snapshot['inflation_rate']}%, trend is {snapshot['inflation_trend']}
- Unemployment: {snapshot['unemployment_rate']}%, trend is {snapshot['unemployment_trend']}
- Industrial Production trend: {snapshot['industrial_production_trend']}
- Yield Curve: {snapshot['yield_curve_status']} (spread: {snapshot['yield_curve_spread']} percentage points)

Write a short, clear explanation (4-6 sentences) covering:
1. Overall recession risk assessment
2. The 2-3 key indicators driving that assessment
3. What a business leader should take away from this

Do not use markdown headers. Write it as flowing paragraph(s), business-report tone.
"""
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text
'''

with open('/content/drive/MyDrive/fred-economic-agent/src/genai_explain.py', 'w') as f:
    f.write(explain_code)

print("Saved to src/genai_explain.py")

Saved to src/genai_explain.py


In [8]:
# --- Executive Economic Report generator ---
def generate_executive_report(snapshot, client):
    prompt = f"""You are an economic analyst preparing a monthly executive report for
company leadership. Use ONLY the verified data below — do not invent any numbers,
statistics, or claims not present in this snapshot.

ECONOMIC SNAPSHOT (as of {snapshot['date']}):
- Recession Risk: {snapshot['recession_risk']} (model probability: {snapshot['recession_probability']*100:.1f}%)
- Inflation: {snapshot['inflation_rate']}%, trend is {snapshot['inflation_trend']}
- Unemployment: {snapshot['unemployment_rate']}%, trend is {snapshot['unemployment_trend']}
- Industrial Production trend: {snapshot['industrial_production_trend']}
- Yield Curve: {snapshot['yield_curve_status']} (spread: {snapshot['yield_curve_spread']} percentage points)

Write a MONTHLY ECONOMIC INTELLIGENCE REPORT with these exact section headers:

Executive Summary
Employment
Inflation
Interest Rates & Yield Curve
Industrial Activity
ML Assessment
Key Risks
Key Indicators to Monitor

Keep each section to 1-3 sentences. Under "Key Risks" and "Key Indicators to Monitor",
use a numbered list of up to 3 items each, grounded only in the data given.
"""
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt
    )
    return response.text

report = generate_executive_report(snapshot, client)
print(report)

# MONTHLY ECONOMIC INTELLIGENCE REPORT

### Executive Summary
As of July 1, 2026, the macroeconomic environment presents a low risk of recession alongside positive momentum in production and employment. However, inflation currently stands at 3.3% with a rising trend that requires attention. Overall economic stability remains intact, supported by a normal yield curve spread of 0.38 percentage points.

### Employment
The current unemployment rate stands at 4.1%. The trajectory for unemployment is falling, reflecting ongoing labor market strength.

### Inflation
Inflation is currently measured at 3.3%. The primary trend for inflation is rising.

### Interest Rates & Yield Curve
The yield curve is currently normal. The yield curve spread is measured at 0.38 percentage points.

### Industrial Activity
Industrial activity demonstrates positive growth momentum. The overall trend for industrial production is rising.

### ML Assessment
The machine learning model assesses the probability of a re

In [9]:
# --- Save executive report generator to src/genai_report.py ---
report_code = '''
def generate_executive_report(snapshot, client):
    prompt = f"""You are an economic analyst preparing a monthly executive report for
company leadership. Use ONLY the verified data below — do not invent any numbers,
statistics, or claims not present in this snapshot.

ECONOMIC SNAPSHOT (as of {snapshot['date']}):
- Recession Risk: {snapshot['recession_risk']} (model probability: {snapshot['recession_probability']*100:.1f}%)
- Inflation: {snapshot['inflation_rate']}%, trend is {snapshot['inflation_trend']}
- Unemployment: {snapshot['unemployment_rate']}%, trend is {snapshot['unemployment_trend']}
- Industrial Production trend: {snapshot['industrial_production_trend']}
- Yield Curve: {snapshot['yield_curve_status']} (spread: {snapshot['yield_curve_spread']} percentage points)

Write a MONTHLY ECONOMIC INTELLIGENCE REPORT with these exact section headers:

Executive Summary
Employment
Inflation
Interest Rates & Yield Curve
Industrial Activity
ML Assessment
Key Risks
Key Indicators to Monitor

Keep each section to 1-3 sentences. Under "Key Risks" and "Key Indicators to Monitor",
use a numbered list of up to 3 items each, grounded only in the data given.
"""
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text
'''

with open('/content/drive/MyDrive/fred-economic-agent/src/genai_report.py', 'w') as f:
    f.write(report_code)

print("Saved to src/genai_report.py")

Saved to src/genai_report.py
